In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
# Metadata
PIPELINE_NAME = "Silver_Order_Items_Load"
RUN_ID = generate_run_id()
START_TIME = start_pipeline()

Pipeline Started : 2026-07-18 12:16:09.823331


In [0]:
SOURCE_TABLE = TARGET_TABLE_ITEM
TARGET_TABLE = SILVER_ORDER_ITEMS
QUARANTINE_TABLE = QUARANTINE_ORDER_ITEMS

In [0]:
# Read Bronze Order Items
order_items_df = spark.table(SOURCE_TABLE)

In [0]:
total_rows = order_items_df.count()

In [0]:
# Dataset Profile

print("ORDER ITEMS DATASET PROFILE")
print(f"Total Records : {total_rows}")
order_items_df.printSchema()
display(order_items_df.limit(10))

ORDER ITEMS DATASET PROFILE
Total Records : 86328
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_date: date (nullable = true)
 |-- pipeline_name: string (nullable = true)
 |-- run_id: string (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp,ingestion_date,pipeline_name,run_id
ORD_0000001,1,PROD_001950,SELL_0224,2023-06-18T14:30:00.000Z,1754.7,20.17,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000002,1,PROD_000989,SELL_0405,2021-06-14T11:02:00.000Z,2105.2,45.48,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000003,1,PROD_000254,SELL_0283,2022-04-03T19:38:00.000Z,1627.94,78.29,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000004,1,PROD_001327,SELL_0437,2021-09-12T22:29:00.000Z,65.22,31.08,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000004,2,PROD_002714,SELL_0396,2021-09-12T22:29:00.000Z,946.39,44.43,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000004,3,PROD_001507,SELL_0446,2021-09-12T22:29:00.000Z,157.96,55.93,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000005,1,PROD_001872,SELL_0152,2022-08-28T07:46:00.000Z,1681.83,14.0,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000006,1,PROD_001518,SELL_0490,2023-05-29T16:27:00.000Z,856.74,64.83,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000007,1,PROD_001504,SELL_0324,2021-09-22T15:50:00.000Z,373.8,66.88,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e
ORD_0000008,1,PROD_001133,SELL_0241,2021-01-21T14:24:00.000Z,899.98,23.95,2026-07-15T07:38:12.114Z,2026-07-15,Bronze_Order_Items,89cd67fb-0173-4983-9c48-0c154f057a6e


# Data Quality Checks

In [0]:
# Primary Key Validation

distinct_primary_keys = (
    order_items_df
    .select("order_id", "order_item_id")
    .distinct()
    .count()
)

primary_key_valid = (distinct_primary_keys == total_rows)

print(f"Distinct Composite Keys : {distinct_primary_keys}")
print(f"Primary Key Validation  : {primary_key_valid}")

Distinct Composite Keys : 86328
Primary Key Validation  : True


In [0]:
# Duplicate validation 
duplicate_rows = duplicate_summary(order_items_df, total_rows)

Duplicate Rows : 0


In [0]:
# Checking nulls 
null_summary(order_items_df)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp,ingestion_date,pipeline_name,run_id
0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Bussiness Rule Validation 

invalid_business_rules = order_items_df.filter(
    (F.col("price") <= 0) |
    (F.col("freight_value") < 0) |
    (F.col("shipping_limit_date").isNull())
).withColumn(
    "quarantine_reason",
    F.when(F.col("price") <= 0,
           "Invalid Price")
     .when(F.col("freight_value") < 0,
           "Negative Freight Value")
     .when(F.col("shipping_limit_date").isNull(),
           "Missing Shipping Limit Date")
).withColumn("validation_type", F.lit("Business Rule"))

business_rule_count = invalid_business_rules.count()
print(f"Business Rule Violations : {business_rule_count}")

Business Rule Violations : 0


In [0]:
# Foreign Key Validation - Orders
silver_orders = spark.table(SILVER_ORDERS)

orphan_items_order_fk = (
    order_items_df.join(
        silver_orders.select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .withColumn("quarantine_reason", F.lit("Order ID Not Found in Silver Orders"))
    .withColumn("validation_type", F.lit("Foreign Key - Order"))
)
print(f"Orphan Items (order_id FK) : {orphan_items_order_fk.count()}")

Orphan Items (order_id FK) : 0


In [0]:
# Foreign Key Validation - Products
silver_products = spark.table(SILVER_PRODUCTS)

orphan_items_product_fk = (
    order_items_df.join(
        silver_products.select("product_id"),
        on="product_id",
        how="left_anti"
    )
    .withColumn("quarantine_reason", F.lit("Product ID Not Found in Silver Products"))
    .withColumn("validation_type", F.lit("Foreign Key - Product"))
)
print(f"Orphan Items (product_id FK) : {orphan_items_product_fk.count()}")

Orphan Items (product_id FK) : 0


In [0]:
# Invalid Order Items (combined - all 3 sources)
invalid_items = (
    invalid_business_rules
    .unionByName(orphan_items_order_fk)
    .unionByName(orphan_items_product_fk)
    .dropDuplicates(["order_id", "order_item_id"])
)
print(f"Total Invalid Order Items : {invalid_items.count()}")

Total Invalid Order Items : 0


In [0]:
valid_items = (
    order_items_df.join(
        invalid_items.select("order_id", "order_item_id"),
        on=["order_id", "order_item_id"],
        how="left_anti"
    )
)
print(f"Valid Order Items : {valid_items.count()}")

Valid Order Items : 86328


In [0]:
valid_items = add_audit_columns(valid_items, PIPELINE_NAME, RUN_ID)

table_exists = spark.catalog.tableExists(TARGET_TABLE)

if not table_exists:
    valid_items.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
    execution_status = "SUCCESS"
else:
    from delta.tables import DeltaTable
    delta_table = DeltaTable.forName(spark, TARGET_TABLE)
    (
        delta_table.alias("target")
        .merge(
            valid_items.alias("source"),
            "target.order_id = source.order_id AND target.order_item_id = source.order_item_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    execution_status = "SUCCESS"

In [0]:
(
    invalid_items.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)
print(f"Quarantined Records : {invalid_items.count()}")

Quarantined Records : 0


In [0]:
silver_items = spark.table(TARGET_TABLE)
rows_written = silver_items.count()

bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_TABLE,
    target=TARGET_TABLE,
    rows_read=total_rows,
    rows_written=rows_written,
    duplicate_count=duplicate_rows,
    start_time=START_TIME,
    status=execution_status
)

LOAD REPORT
Pipeline        : Silver_Order_Items_Load
Run ID          : a6cedcf3-ad0c-44fb-9c6c-d7017b85e6d9
Source          : retailmart.bronze.order_items
Target          : retailmart.silver.order_items
Rows Read       : 86328
Rows Written    : 86328
Duplicate Rows  : 0
Start Time      : 2026-07-18 12:16:09.823331
End Time        : 2026-07-18 12:16:34.112519
Duration (sec)  : 24.29
Status          : SUCCESS


# Engineering Observations

- Validated composite primary key (order_id, payment_sequential).
- Performed duplicate and null value analysis.
- Standardized payment_type using trim() and lowercase().
- Applied business rule validations for payment value, installments, payment sequence, and payment type.
- Validated foreign key relationship with Silver Orders.
- Invalid records were isolated and stored in the quarantine table.
- Audit metadata was refreshed before loading.
- Implemented incremental loading using Delta Lake SCD Type 1 MERGE.
- Conditional updates ensure only changed records are updated, reducing unnecessary writes.